# บทที่ 4 — Descriptive Statistics: สถานภาพการวิจัยภายใต้แผนงาน อพท.

Notebook นี้จัดลำดับการวิเคราะห์ตามประเด็นในเอกสาร Word ตั้งแต่ **4.1–4.7** และใช้ข้อมูลจาก Excel เฉพาะชีต **`สถานภาพแผนงาน -Clean`**

**หลักการแสดงผล**
- จำนวนแสดงเป็น `n (%)`
- Multiple response ใช้จำนวนโครงการทั้งหมดเป็นตัวหาร จึงรวมร้อยละอาจเกิน 100%
- โครงการที่ใช้วิเคราะห์ถูกคัดจากแถวที่มีทั้ง **งบประมาณ** และ **สถานะงาน** เพื่อกันแถวบันทึกท้ายชีตออก
- ไม่แก้ไขไฟล์ Excel ต้นฉบับ
- กราฟบันทึกอัตโนมัติในโฟลเดอร์ `chapter4_outputs/`


**Version 2:** เพิ่ม Publication-ready chart configuration สำหรับแก้ display label, wrap ข้อความ, สี, ขนาดภาพ และลำดับหมวด โดยไม่เปลี่ยนค่าจริงใน Excel

In [ ]:
# 0) Imports & configuration
from pathlib import Path
import re
import math
import warnings
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=UserWarning)

SHEET_NAME = "สถานภาพแผนงาน -Clean"
OUTPUT_DIR = Path("chapter4_outputs_v3")
OUTPUT_DIR.mkdir(exist_ok=True)

# รองรับทั้งกรณีเปิด notebook ในโฟลเดอร์เดียวกับ Excel และใน /mnt/data
candidates = [
    Path("ตารางสถานภาพงานวิจัย-final05Sep2026.xlsx"),
    Path("/mnt/data/ตารางสถานภาพงานวิจัย-final05Sep2026.xlsx"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "ไม่พบไฟล์ Excel กรุณาวาง 'ตารางสถานภาพงานวิจัย-final05Sep2026.xlsx' "
        "ไว้ในโฟลเดอร์เดียวกับ notebook"
    )

# เลือก font ไทยที่มีในเครื่องโดยอัตโนมัติ
preferred_fonts = ["Noto Sans Thai", "Tahoma", "Leelawadee UI", "Arial Unicode MS", "DejaVu Sans"]
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
thai_font = next((f for f in preferred_fonts if f in available_fonts), "DejaVu Sans")

plt.rcParams.update({
    "font.family": [thai_font, "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "axes.unicode_minus": False,
})

print("DATA:", DATA_PATH)
print("SHEET:", SHEET_NAME)
print("FONT:", thai_font)




## 1) อ่านข้อมูลและเตรียมชุดวิเคราะห์

ไฟล์นี้มีหัวตาราง 2 ชั้น จึงอ่านด้วย `header=[0,1]` แล้วใช้ชื่อหัวข้อชั้นล่างเป็นชื่อคอลัมน์หลัก  
การคัด 27 โครงการจะอาศัยแถวที่มี **งบประมาณที่ได้รับจัดสรร** และ **สถานะงาน** ไม่เป็นค่าว่าง


In [ ]:
raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, header=[0, 1])

def flatten_columns(columns):
    names = []
    used = {}
    for top, bottom in columns:
        bottom = str(bottom)
        top = str(top)
        base = top.strip() if bottom.startswith("Unnamed:") else bottom.strip()
        base = re.sub(r"\s+", " ", base)
        count = used.get(base, 0)
        used[base] = count + 1
        names.append(base if count == 0 else f"{base}__{count+1}")
    return names

raw.columns = flatten_columns(raw.columns)

# แปลงงบประมาณเป็นตัวเลขเพื่อใช้เป็นเงื่อนไขคัดแถวโครงการจริง
raw["งบประมาณที่ได้รับจัดสรร"] = pd.to_numeric(
    raw["งบประมาณที่ได้รับจัดสรร"], errors="coerce"
)

df = raw[
    raw["งบประมาณที่ได้รับจัดสรร"].notna()
    & raw["สถานะงาน"].notna()
].copy()

df = df.reset_index(drop=True)

print(f"จำนวนแถวในชีตทั้งหมด: {len(raw):,}")
print(f"จำนวนโครงการที่ใช้วิเคราะห์: {len(df):,}")
display(df[["รหัสโครงการ", "ชื่อโครงการภาษาไทย", "งบประมาณที่ได้รับจัดสรร", "สถานะงาน"]].head())


## 2) Chart configuration — ปรับชื่อ สี และสัดส่วนรูปจากจุดเดียว

ส่วนนี้ออกแบบให้ **ค่าจริงสำหรับการวิเคราะห์ไม่ถูกแก้** แต่สามารถเปลี่ยนชื่อที่แสดงบนกราฟ (`display label`) ได้อิสระ

วิธีใช้หลัก ๆ:
- แก้คำยาวใน `*_LABELS` ให้เป็นคำสั้นที่ต้องการแสดง
- ปรับสีใน `COLORS`
- ปรับขนาดตัวอักษร/ความละเอียดใน `CHART_STYLE`
- ถ้าชื่อยังยาว ให้ปรับ `wrap_width` ตอนเรียก `barh_count()`

> ตารางและการคำนวณยังใช้ค่าต้นฉบับจาก Excel เสมอ การย่อชื่อมีผลเฉพาะรูปเท่านั้น


In [ ]:
# ============================================================
# PUBLICATION-READY CHART CONFIGURATION
# แก้ตรงนี้ก่อนสร้างรูปได้เลย โดยไม่กระทบค่าคำนวณจริง
# ============================================================

COLORS = {
    "size": "#8FB9E0",         # ฟ้าอ่อน
    "status": "#A8D5BA",       # เขียวอ่อน
    "duration": "#F6C28B",     # ส้มพีชอ่อน
    "affiliation": "#C7B6E5",  # ม่วงอ่อน
    "team": "#F2B5B5",         # ชมพูอ่อน
    "pmu": "#9ED9CC",          # เขียวมิ้นต์
    "research_type": "#F7D6A3",# ครีมส้มอ่อน
    "oecd": "#AFCBFF",         # ฟ้านม
    "output": "#F4B6C2",       # ชมพูพาสเทล
    "trl": "#C9B6E4",          # ม่วงลาเวนเดอร์
    "srl": "#D9C2F0",          # ม่วงอ่อนมาก
    "academic": "#A7C7E7",     # ฟ้าเทาอ่อน
    "actual_users": "#B7D88C", # เขียวใบไม้อ่อน
    "target_users": "#A2D2A2", # เขียวเซจอ่อน
    "outcome": "#F7C59F",      # พีชอ่อน
    "impact": "#F5A97F",       # ส้มอ่อน
    "primary": "#8FB9E0",      # alias เดิม
    "secondary": "#A2D2A2",    # alias เดิม
    "accent": "#F5A97F",       # alias เดิม
    "purple": "#C9B6E4",       # alias เดิม
    "neutral": "#D9D9D9",      # เทาอ่อน
    "dark": "#4A4A4A",
}

CHART_COLOR_MAP = {
    "4_2_1_project_size_count.png": COLORS["size"],
    "4_2_2_close_status.png": COLORS["status"],
    "4_2_2_duration_groups.png": COLORS["duration"],
    "4_2_3_pi_affiliations.png": COLORS["affiliation"],
    "4_2_3_researcher_team_size.png": COLORS["team"],
    "4_3_2_pmu_research_framework.png": COLORS["pmu"],
    "4_3_2_research_type.png": COLORS["research_type"],
    "4_3_3_oecd_research_field.png": COLORS["oecd"],
    "4_4_1_output_types.png": COLORS["output"],
    "4_4_2_trl.png": COLORS["trl"],
    "4_4_3_srl.png": COLORS["srl"],
    "4_5_1_academic_benefits.png": COLORS["academic"],
    "4_6_1_1_actual_users.png": COLORS["actual_users"],
    "4_6_1_1_target_users.png": COLORS["target_users"],
    "4_6_1_2_outcome_types.png": COLORS["outcome"],
    "4_6_2_impact_dimensions.png": COLORS["impact"],
}


CHART_STYLE = {
    "title_size": 16,
    "axis_label_size": 12,
    "tick_size": 11,
    "annotation_size": 11,
    "grid_alpha": 0.14,
    "save_dpi": 240,
    "default_width": 10,
    "default_height": 5.5,
    "bar_height_per_item": 0.55,
}

# ------------------------------------------------------------------
# Display labels: ซ้าย = ค่าจริงใน Excel / ขวา = คำสั้นสำหรับรูป
# ถ้าหา key ไม่เจอ ระบบจะใช้ข้อความเดิมและ wrap ให้อัตโนมัติ
# ------------------------------------------------------------------

PMU_LABELS = {
    "การพัฒนากลไกการจัดบริการสาธารณะขององค์กรปกครองส่วนท้องถิ่น (User คือ ช่วยชาวบ้านในพื้นที่)": "พัฒนากลไกบริการสาธารณะของ อปท.",
    "การพัฒนาเทคโนโลยีดิจิทัล (User คือ อปท. เพื่อพัฒนาการทำงาน)": "พัฒนาเทคโนโลยีดิจิทัลของ อปท.",
    "การพัฒนากลไกและกระบวนการสร้างการเปลี่ยนแปลงเพื่อเพิ่มรายได้ของท้องถิ่น": "กลไกเพิ่มรายได้ของท้องถิ่น",
    "การพัฒนาศักยภาพเชิงสถาบันและกรอบกฎหมาย": "ศักยภาพเชิงสถาบันและกฎหมาย",
    "เป็นการพัฒนาสมรรถนะ?การบริหารจัดการน้ำ ซึ่งเป็นบริการสาธารณะของ อปท.": "สมรรถนะการบริหารจัดการน้ำ",
    "เป็นการพัฒนาศักยภาพนักวิจัย เพื่อตอบสนองต่อการพัฒนาปสก.การทำงานของ อปท.": "พัฒนาศักยภาพนักวิจัยเพื่อ อปท.",
    "การพัฒนาเทคโนโลยีดิจิทัล": "พัฒนาเทคโนโลยีดิจิทัล",
}

OUTPUT_LABELS = {
    "ฐานข้อมูล ระบบและกลไก": "ฐานข้อมูล ระบบและกลไก",
    "กำลังคน หรือหน่วยงาน ที่ได้รับการพัฒนาทักษะ": "กำลังคน/หน่วยงานที่ได้รับการพัฒนาทักษะ",
    "ข้อเสนอแนะเชิงนโยบาย:ข้อเสนอที่มุ่งใช้ประกอบการตัดสินใจ การกำหนดนโยบาย มาตรการ แผนงาน แนวทางปฏิบัติ หรือกฎเกณฑ์ของหน่วยงานหรือองค์กร": "ข้อเสนอแนะเชิงนโยบาย",
    "เครื่องมือ และโครงสร้างพื้นฐานที่สร้างขึ้น หรือพัฒนาต่อยอดภายใต้โครงการ": "เครื่องมือ/โครงสร้างพื้นฐาน",
}

USER_LABELS = {
    "หน่วยงานภาครัฐในระดับพื้นที่ ตำบล อำเภอ จังหวัด": "หน่วยงานภาครัฐระดับพื้นที่",
    "หน่วยงานรัฐระดับส่วนกลาง/ผู้กำหนดนโยบาย": "หน่วยงานส่วนกลาง/ผู้กำหนดนโยบาย",
    "บุคลากรทางการศึกษา/สถาบันการศึกษา": "บุคลากร/สถาบันการศึกษา",
}

OUTCOME_LABELS = {
    "ลดความสูญเสียทางเศรษฐกิจสังคม เช่น ลดอัตราการเจ็บป่วย ลดอัตราการตาย และลดค่าใช้จ่ายด้านสุขภาพ เป็นต้น": "ลดความสูญเสียทางเศรษฐกิจและสังคม",
    "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต/เพิ่มประสิทธิภาพการดำเนินงาน": "เพิ่มผลิตภาพ/ประสิทธิภาพการดำเนินงาน",
    "พัฒนาคุณภาพชีวิต (เพราะเข้าถึงบริการสุขภาพคุณภาพมาตรฐาน)": "พัฒนาคุณภาพชีวิต",
    "คุณภาพสิ่งแวดล้อมดีขึ้น ลดมลพิษ/มลภาวะ/ลดก๊าซเรือนกระจก": "คุณภาพสิ่งแวดล้อมดีขึ้น",
}

ACADEMIC_LABELS = {
    "จำนวนครั้งการเผยแพร่ผ่านการอบรม/สัมมนา/เวทีสาธารณะ/นิทรรศการ": "อบรม/สัมมนา/เวทีสาธารณะ/นิทรรศการ",
    "จำนวนสื่อ clip vdo หรือเพจเผยแพร่/งานเขียนออนไลน์": "สื่อ/คลิปวิดีโอ/เพจ/งานเขียนออนไลน์",
    "จำนวนครั้งการเผยแพร่ผ่าน วิดีทัศน์ โทรทัศน์ วิทยุ นสพ. อินเตอร์เน็ต": "เผยแพร่ผ่านสื่อมวลชน/อินเทอร์เน็ต",
    "มีการใช้ประโยชน์กับการเรียนการสอน (จำนวนวิชา)": "ใช้ประโยชน์ในการเรียนการสอน",
    "มีการใช้ประโยชน์กับการวิจัยเพื่อพัฒนานิสิต (จำนวนนิสิต ที่ทำวิจัยหรือวิทยานิพนธ์)": "ใช้พัฒนางานวิจัย/วิทยานิพนธ์นิสิต",
    "จำนวนบทความ (นำเสนอในที่ประชุมระดับนานาชาติ)": "บทความนำเสนอประชุมระดับนานาชาติ",
    "จำนวนบทความ (นำเสนอในที่ประชุมระดับประเทศ)": "บทความนำเสนอประชุมระดับประเทศ",
}

OECD_FIELD_LABELS = {
    "5. สังคมศาสตร์ (Social Sciences)": "สังคมศาสตร์",
    "2. วิศวกรรมและเทคโนโลยี (Engineering and technology)": "วิศวกรรมและเทคโนโลยี",
    "3. วิทยาศาสตร์การแพทย์และสุขภาพ (Medical and Health Sciences)": "วิทยาศาสตร์การแพทย์และสุขภาพ",
}

# กรณีอยากแก้ชื่อสังกัดเฉพาะแห่ง ให้เติมใน dict นี้
AFFILIATION_LABELS = {
    "โรงพยาบาลวชิระ(วชิรพยาบาล) (ย้ายไปใต้คณะแพทยศาสตร์วชิรพยาบาล มหาวิทยาลัยนวมินทราธิราช)": "วชิรพยาบาล ม.นวมินทราธิราช",
}


def shorten_display_label(value, label_map=None, wrap_width=30):
    """คืนชื่อสำหรับกราฟเท่านั้น: map ก่อน แล้วค่อยตัดบรรทัดให้พอดี"""
    text = str(value)
    if label_map:
        text = label_map.get(text, text)
    if wrap_width:
        text = "\n".join(textwrap.wrap(text, width=wrap_width, break_long_words=False, break_on_hyphens=False))
    return text


def apply_display_labels(index, label_map=None, wrap_width=30):
    return [shorten_display_label(x, label_map=label_map, wrap_width=wrap_width) for x in index]




In [ ]:
# Helper functions สำหรับตารางและกราฟ
def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace("\n", " ").replace("\r", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else np.nan
def n_pct(n, denom):
    if denom == 0:
        return f"{int(n):,} (0.0%)"
    return f"{int(n):,} ({n/denom*100:.1f}%)"
def frequency_table(series, denom=None, dropna=True, sort=True):
    s = series.map(clean_text)
    counts = s.value_counts(dropna=dropna)
    if denom is None:
        denom = s.notna().sum()
    out = pd.DataFrame({"จำนวน": counts})
    out["ร้อยละ"] = out["จำนวน"] / denom * 100
    out["n (%)"] = [n_pct(n, denom) for n in out["จำนวน"]]
    if sort:
        out = out.sort_values(["จำนวน"], ascending=False)
    return out
def barh_count(
    table,
    title,
    xlabel="จำนวนโครงการ",
    filename=None,
    pct_col="ร้อยละ",
    *,
    label_map=None,
    wrap_width=24,
    color=None,
    figsize=None,
    order=None,
    sort_by_count=True,
    ylim_pad=1.18,
    note=None,
):
    """
    Vertical bar chart
    - แกน X = หมวดหมู่
    - แกน Y = จำนวน
    - แต่ละแท่งใช้สี pastel แตกต่างกัน
    """

    plot_df = table.copy()

    # ---------------------------------------------------------
    # จัดลำดับข้อมูล
    # ---------------------------------------------------------
    if order is not None:
        existing = [x for x in order if x in plot_df.index]
        remainder = [x for x in plot_df.index if x not in existing]
        plot_df = plot_df.reindex(existing + remainder)

    elif sort_by_count:
        plot_df = plot_df.sort_values(
            "จำนวน",
            ascending=False
        )

    # ---------------------------------------------------------
    # ชื่อที่ใช้แสดงบนกราฟ
    # ไม่แก้ข้อมูลต้นฉบับ
    # ---------------------------------------------------------
    display_index = apply_display_labels(
        plot_df.index,
        label_map=label_map,
        wrap_width=wrap_width
    )

    # ---------------------------------------------------------
    # ขนาดภาพ
    # ---------------------------------------------------------
    n_cat = len(plot_df)

    if figsize is None:
        width = max(
            9,
            min(18, n_cat * 1.2 + 5)
        )

        height = 7

        figsize = (width, height)

    # ---------------------------------------------------------
    # สี Pastel
    # ---------------------------------------------------------
    PASTEL_BAR_COLORS = [
        "#8FB9E0",   # ฟ้าอ่อน
        "#A8D5BA",   # เขียวอ่อน
        "#F6C28B",   # พีช
        "#C7B6E5",   # ม่วงอ่อน
        "#F2B5B5",   # ชมพู
        "#9ED9CC",   # มิ้นต์
        "#F7D6A3",   # ครีมส้ม
        "#AFCBFF",   # ฟ้านม
        "#F4B6C2",   # ชมพูอ่อน
        "#B7D88C",   # เขียวอ่อน
        "#D9C2F0",   # ลาเวนเดอร์
        "#FFD6A5",   # apricot
    ]

    # ให้แต่ละแท่งใช้สีต่างกัน
    bar_colors = [
        PASTEL_BAR_COLORS[
            i % len(PASTEL_BAR_COLORS)
        ]
        for i in range(n_cat)
    ]

    # ---------------------------------------------------------
    # สร้างกราฟ
    # ---------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=figsize
    )

    bars = ax.bar(
        range(n_cat),
        plot_df["จำนวน"].values,
        color=bar_colors,
        edgecolor="white",
        linewidth=1
    )

    # ---------------------------------------------------------
    # Title
    # ---------------------------------------------------------
    ax.set_title(
        title,
        fontsize=CHART_STYLE.get(
            "title_size",
            18
        ),
        pad=15,
        fontweight="bold"
    )

    # ---------------------------------------------------------
    # Axis
    # ---------------------------------------------------------
    ax.set_ylabel(
        xlabel,
        fontsize=CHART_STYLE.get(
            "axis_label_size",
            13
        )
    )

    ax.set_xlabel("")

    ax.set_xticks(
        range(n_cat)
    )

    ax.set_xticklabels(
        display_index,
        fontsize=CHART_STYLE.get(
            "tick_size",
            11
        )
    )

    # ---------------------------------------------------------
    # หมุน label ถ้าข้อความยาว
    # ---------------------------------------------------------
    max_label_len = max(
        [
            len(
                str(x).replace(
                    "\n",
                    ""
                )
            )
            for x in display_index
        ]
    ) if n_cat > 0 else 0

    if (
        n_cat >= 6
        or max_label_len > 15
    ):

        plt.setp(
            ax.get_xticklabels(),
            rotation=25,
            ha="right"
        )

    else:

        plt.setp(
            ax.get_xticklabels(),
            rotation=0,
            ha="center"
        )

    # ---------------------------------------------------------
    # Grid
    # ---------------------------------------------------------
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.20
    )

    ax.set_axisbelow(True)

    ax.spines[
        ["top", "right"]
    ].set_visible(False)

    # ---------------------------------------------------------
    # Scale Y
    # ---------------------------------------------------------
    max_val = max(
        float(
            plot_df["จำนวน"].max()
        ),
        1
    )

    ax.set_ylim(
        0,
        max_val * ylim_pad + 0.5
    )

    # ---------------------------------------------------------
    # จำนวน + %
    # บนยอดแท่ง
    # ---------------------------------------------------------
    for bar, (_, row) in zip(
        bars,
        plot_df.iterrows()
    ):

        n = int(
            row["จำนวน"]
        )

        if pct_col in row.index:

            label = (
                f"{n} "
                f"({row[pct_col]:.1f}%)"
            )

        else:

            label = str(n)

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            bar.get_height()
            + max_val * 0.02,

            label,

            ha="center",
            va="bottom",

            fontsize=CHART_STYLE.get(
                "annotation_size",
                11
            )
        )

    # ---------------------------------------------------------
    # Note ด้านล่าง
    # ---------------------------------------------------------
    if note:

        fig.text(
            0.01,
            0.01,
            note,
            ha="left",
            va="bottom",
            fontsize=9,
            color="#555555"
        )

        fig.tight_layout(
            rect=[
                0,
                0.05,
                1,
                1
            ]
        )

    else:

        fig.tight_layout()

    # ---------------------------------------------------------
    # Save
    # ---------------------------------------------------------
    if filename:

        fig.savefig(
            OUTPUT_DIR / filename,
            dpi=CHART_STYLE.get(
                "save_dpi",
                240
            ),
            bbox_inches="tight"
        )

    plt.show()

    return fig
def save_table(table, filename):
    table.to_csv(OUTPUT_DIR / filename, encoding="utf-8-sig")
def numeric_prefix(series):
    """ดึงเลขนำหน้าจากข้อความ เช่น '6. ต้นแบบ...' -> 6"""
    return pd.to_numeric(
        series.astype(str).str.extract(r"^\s*([0-9]+)", expand=False),
        errors="coerce"
    )
N = len(df)
print("N =", N)



# 4.1 ข้อมูลเบื้องต้นของแผนงานวิจัย

ส่วน 4.1 ใน Word เป็นบริบทเชิงนโยบาย ได้แก่ นิยามแผน/แพลตฟอร์ม/โปรแกรม, OKR และโครงสร้างแผนงาน  
Notebook จึงใช้ส่วนนี้เป็น **บริบทประกอบ** และเริ่ม descriptive statistics เชิงปริมาณเต็มรูปแบบตั้งแต่ 4.2

> ตัวเลขภาพรวมด้านจำนวนโครงการและงบประมาณตรวจจากฐานข้อมูลอีกครั้งด้านล่าง


In [ ]:
# 4.1 ภาพรวมเชิงตัวเลข
overview = pd.DataFrame({
    "ตัวชี้วัด": [
        "จำนวนโครงการทั้งหมด",
        "งบประมาณรวม (บาท)",
        "งบประมาณเฉลี่ยต่อโครงการ (บาท)",
        "ค่ามัธยฐานงบประมาณต่อโครงการ (บาท)"
    ],
    "ค่า": [
        N,
        df["งบประมาณที่ได้รับจัดสรร"].sum(),
        df["งบประมาณที่ได้รับจัดสรร"].mean(),
        df["งบประมาณที่ได้รับจัดสรร"].median()
    ]
})
display(overview)


# 4.2 ปัจจัยนำเข้า

ตามกรอบ Word ส่วนนี้ **คิดรวมโครงการที่ขยายเวลาและยุติ** เพื่อสะท้อนทรัพยากรทั้งหมดที่ใช้ขับเคลื่อนแผนงาน


## 4.2.1 งบประมาณ

กราฟแนะนำ: **horizontal bar chart** เพราะอ่านความแตกต่างระหว่างขนาดโครงการได้ง่าย และสามารถใส่ทั้ง `จำนวน (%)` กับงบเฉลี่ยได้โดยไม่แน่นเกินไป


In [ ]:
budget = df["งบประมาณที่ได้รับจัดสรร"]

budget_desc = pd.Series({
    "งบประมาณรวม (ล้านบาท)": budget.sum()/1e6,
    "งบประมาณเฉลี่ย (ล้านบาท/โครงการ)": budget.mean()/1e6,
    "มัธยฐาน (ล้านบาท/โครงการ)": budget.median()/1e6,
    "ต่ำสุด (ล้านบาท)": budget.min()/1e6,
    "สูงสุด (ล้านบาท)": budget.max()/1e6,
})
display(budget_desc.to_frame("ค่า").round(3))

size_col = "ขนาดโครงการ"
size_table = (
    df.groupby(size_col, dropna=False)
      .agg(
          n_projects=("รหัสโครงการ", "count"),
          budget_total=("งบประมาณที่ได้รับจัดสรร", "sum"),
          budget_mean=("งบประมาณที่ได้รับจัดสรร", "mean"),
      )
      .rename(columns={"n_projects":"จำนวน", "budget_total":"งบประมาณรวม", "budget_mean":"งบประมาณเฉลี่ย"})
)
size_table["ร้อยละ"] = size_table["จำนวน"] / N * 100
size_table["สัดส่วนงบประมาณ"] = size_table["งบประมาณรวม"] / budget.sum() * 100
size_table["n (%)"] = [n_pct(n, N) for n in size_table["จำนวน"]]
size_table["งบรวม (ล้านบาท)"] = size_table["งบประมาณรวม"]/1e6
size_table["งบเฉลี่ย (ล้านบาท)"] = size_table["งบประมาณเฉลี่ย"]/1e6

display(size_table[["n (%)", "งบรวม (ล้านบาท)", "สัดส่วนงบประมาณ", "งบเฉลี่ย (ล้านบาท)"]].round(2))
save_table(size_table, "4_2_1_budget_by_project_size.csv")

barh_count(
    size_table[["จำนวน", "ร้อยละ"]],
    "จำนวนโครงการจำแนกตามขนาดโครงการ",
    filename="4_2_1_project_size_count.png",
    color=COLORS["primary"],
    wrap_width=24
)


## 4.2.2 ระยะเวลาและสถานะโครงการ

แสดงทั้ง
1. การกระจายตามระยะเวลา: ต่ำกว่า 1 ปี / 1 ปี / มากกว่า 1 ปี  
2. จำนวนโครงการขยายเวลาและยุติ  
3. สถานะปิดโครงการ โดยคำนวณร้อยละจาก **โครงการที่ไม่ยุติ** ตามกรอบ Word


In [ ]:
# ระยะเวลาใช้ปีและเดือนตามจริง
years = pd.to_numeric(df["ระยะเวลาปี"], errors="coerce").fillna(0)
months = pd.to_numeric(df["ระยะเวลาเดือน"], errors="coerce").fillna(0)
duration_months = years*12 + months

df["ระยะเวลารวม_เดือน"] = duration_months

def duration_group(m):
    if pd.isna(m):
        return "ไม่ระบุ"
    if m < 12:
        return "ต่ำกว่า 1 ปี"
    if m == 12:
        return "1 ปี"
    return "มากกว่า 1 ปี"

df["กลุ่มระยะเวลา"] = df["ระยะเวลารวม_เดือน"].map(duration_group)

duration_order = ["ต่ำกว่า 1 ปี", "1 ปี", "มากกว่า 1 ปี", "ไม่ระบุ"]
duration_table = frequency_table(df["กลุ่มระยะเวลา"], denom=N).reindex(
    [x for x in duration_order if x in set(df["กลุ่มระยะเวลา"])]
)

display(pd.Series({
    "ระยะเวลาเฉลี่ย (เดือน)": df["ระยะเวลารวม_เดือน"].mean(),
    "มัธยฐาน (เดือน)": df["ระยะเวลารวม_เดือน"].median(),
    "ต่ำสุด (เดือน)": df["ระยะเวลารวม_เดือน"].min(),
    "สูงสุด (เดือน)": df["ระยะเวลารวม_เดือน"].max(),
}).to_frame("ค่า").round(2))

display(duration_table[["จำนวน", "ร้อยละ", "n (%)"]])
save_table(duration_table, "4_2_2_duration_groups.csv")

barh_count(
    duration_table[["จำนวน", "ร้อยละ"]],
    "จำนวนโครงการจำแนกตามระยะเวลาดำเนินงาน",
    filename="4_2_2_duration_groups.png",
    color=COLORS["primary"],
    wrap_width=24
)

# ขยายเวลา
extension_months = pd.to_numeric(df["ระยะเวลาที่ขยายเวลา (month) (Gift)"], errors="coerce").fillna(0)
extended = extension_months.gt(0)
terminated = df["สถานะงาน"].astype(str).str.contains("ยุติ", na=False)

print("ขยายเวลา:", n_pct(extended.sum(), N))
print("ยุติโครงการ:", n_pct(terminated.sum(), N))

# ปิดโครงการคำนวณเฉพาะโครงการที่ไม่ยุติ
non_terminated = df.loc[~terminated].copy()
closed = non_terminated["สถานะงาน"].astype(str).str.strip().eq("ปิดโครงการ")
close_table = pd.DataFrame({
    "จำนวน": [closed.sum(), (~closed).sum()]
}, index=["ปิดโครงการ", "ยังไม่ปิดโครงการ"])
close_table["ร้อยละ"] = close_table["จำนวน"] / len(non_terminated) * 100
close_table["n (%)"] = [n_pct(n, len(non_terminated)) for n in close_table["จำนวน"]]

display(close_table)
save_table(close_table, "4_2_2_close_status_nonterminated.csv")

# ============================================================
# 4.2.2 Pie chart 2 วง
# วงที่ 1: สถานะโครงการจาก column AI
# วงที่ 2: รายละเอียดโครงการขยายเวลา
# ============================================================

# ------------------------------------------------------------
# วงที่ 1: ใช้ column AI = สถานะงาน
# แบ่งเป็น:
# 1) ปิดโครงการแล้ว
# 2) ขยายเวลา (อยู่ระหว่างดำเนินการ)
# 3) ยุติโครงการ
# ------------------------------------------------------------

status_series = df["สถานะงาน"].astype(str).str.strip()

closed_n = (status_series == "ปิดโครงการ").sum()
terminated_n = status_series.str.contains("ยุติ", na=False).sum()

# ถือว่าที่เหลือซึ่งไม่ใช่ปิดโครงการและไม่ใช่ยุติ = ขยายเวลา/อยู่ระหว่างดำเนินการ
extended_n = len(df) - closed_n - terminated_n

overall_labels = [
    "ปิดโครงการแล้ว",
    "ขยายเวลา",
    "ยุติโครงการ"
]

overall_sizes = [
    closed_n,
    extended_n,
    terminated_n
]

# ------------------------------------------------------------
# วงที่ 2: เจาะเฉพาะโครงการขยายเวลา
# ใช้จำนวนเดือนจากคอลัมน์ระยะเวลาที่ขยายเวลา
# แยกเป็น:
# - ขยาย 6 เดือน
# - ขยาย 3 เดือน
# ------------------------------------------------------------

extension_months = pd.to_numeric(
    df["ระยะเวลาที่ขยายเวลา (month) (Gift)"],
    errors="coerce"
).fillna(0)

ext_6 = (extension_months == 6).sum()
ext_3 = (extension_months == 3).sum()

ext_labels = []
ext_sizes = []

if ext_6 > 0:
    ext_labels.append("ขยาย 6 เดือน")
    ext_sizes.append(ext_6)

if ext_3 > 0:
    ext_labels.append("ขยาย 3 เดือน")
    ext_sizes.append(ext_3)

# -----------------------------
# pastel colors โทนเดิม
# -----------------------------
overall_colors = [
    "#8FB9E0",  # ฟ้าอ่อน
    "#A8D5BA",  # เขียวอ่อน
    "#F6C28B",  # พีชอ่อน
    "#D9D9D9",  # เทาอ่อน
]

extension_colors = [
    "#C7B6E5",  # ม่วงอ่อน
    "#F2B5B5",  # ชมพูอ่อน
    "#9ED9CC",  # มิ้นต์อ่อน (เผื่อมีมากกว่า 2)
]

# -----------------------------
# ฟังก์ชัน label บน pie
# -----------------------------
def autopct_with_n(values):
    total = sum(values)
    def _fmt(pct):
        n = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n({n})"
    return _fmt

# -----------------------------
# plot 2 วง
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# วงที่ 1: ภาพรวม
wedges1, texts1, autotexts1 = axes[0].pie(
    overall_sizes,
    labels=overall_labels,
    autopct=autopct_with_n(overall_sizes),
    startangle=90,
    colors=overall_colors[:len(overall_sizes)],
    pctdistance=0.72,
    labeldistance=1.08,
    wedgeprops=dict(edgecolor="white", linewidth=1)
)

axes[0].set_title(
    "ภาพรวมระยะเวลาดำเนินงานของโครงการ",
    fontsize=14,
    fontweight="bold",
    pad=14
)

# ทำให้เป็น donut ดูสวยขึ้น
centre_circle1 = plt.Circle((0, 0), 0.45, fc="white")
axes[0].add_artist(centre_circle1)

# วงที่ 2: เจาะการขยายเวลา
if len(ext_sizes) > 0:
    wedges2, texts2, autotexts2 = axes[1].pie(
        ext_sizes,
        labels=ext_labels,
        autopct=autopct_with_n(ext_sizes),
        startangle=90,
        colors=extension_colors[:len(ext_sizes)],
        pctdistance=0.72,
        labeldistance=1.08,
        wedgeprops=dict(edgecolor="white", linewidth=1)
    )

    axes[1].set_title(
        "การขยายเวลาโครงการ",
        fontsize=14,
        fontweight="bold",
        pad=14
    )

    centre_circle2 = plt.Circle((0, 0), 0.45, fc="white")
    axes[1].add_artist(centre_circle2)

else:
    axes[1].text(
        0.5, 0.5,
        "ไม่มีโครงการขยายเวลา",
        ha="center", va="center", fontsize=13
    )
    axes[1].set_title(
        "การขยายเวลาโครงการ",
        fontsize=14,
        fontweight="bold",
        pad=14
    )

# ปรับข้อความบน pie
for t in texts1 + autotexts1:
    t.set_fontsize(10)

if len(ext_sizes) > 0:
    for t in texts2 + autotexts2:
        t.set_fontsize(10)

fig.suptitle(
    "สถานะการดำเนินงานของโครงการวิจัย",
    fontsize=16,
    fontweight="bold",
    y=1.02
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "4_2_2_duration_groups.png",
    dpi=CHART_STYLE.get("save_dpi", 240),
    bbox_inches="tight"
)

plt.show()


## 4.2.3 นักวิจัย คณะผู้วิจัย และหน่วยงานร่วมวิจัย

กราฟที่เหมาะ:
- จำนวนนักวิจัยต่อโครงการ → histogram/bar ตามช่วง
- สังกัดหัวหน้าโครงการ → horizontal bar โดยแสดงเฉพาะหน่วยงานที่มีโครงการ


In [ ]:
# ============================================================
# 4.3.2 ประเภทของการวิจัย + ประเภทตามกรอบ บพท.
# แบบ stacked bar ("ขนมชั้น")
#
# ความสูงแท่ง = จำนวนโครงการ
# ชั้นในแท่ง = ขนาดเล็ก / กลาง / ใหญ่
# annotation ในแต่ละชั้น = จำนวนโครงการ + งบเฉลี่ย (ล้านบาท)
# ============================================================

research_type_col = "ประเภทของการวิจัย"
size_col = "ขนาดโครงการ"
budget_col = "งบประมาณที่ได้รับจัดสรร"

# ------------------------------------------------------------
# normalize ชื่อขนาดโครงการจาก Column K
# ให้ตรงกับภาพ 4.2.1
# ------------------------------------------------------------
def normalize_project_size(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if "เล็ก" in x:
        return "ขนาดเล็ก"
    elif "กลาง" in x:
        return "ขนาดกลาง"
    elif "ใหญ่" in x:
        return "ขนาดใหญ่"
    else:
        return x

df["ขนาดโครงการ_มาตรฐาน"] = df[size_col].apply(normalize_project_size)
size_std_col = "ขนาดโครงการ_มาตรฐาน"

size_order = ["ขนาดเล็ก", "ขนาดกลาง", "ขนาดใหญ่"]

size_colors = {
    "ขนาดเล็ก": "#8FB9E0",   # ฟ้าอ่อน
    "ขนาดกลาง": "#A8D5BA",   # เขียวอ่อน
    "ขนาดใหญ่": "#F6C28B",   # พีชอ่อน
}

# ============================================================
# helper function: stacked bar
# ============================================================
def plot_stacked_count_with_budget(
    count_table,
    budget_table,
    total_pct_table,
    title,
    filename,
    label_map=None,
    wrap_width=24,
    note=None,
    figsize=None,
    rotate_xticks=0
):
    plot_counts = count_table.copy()
    plot_budgets = budget_table.copy()
    plot_totals = total_pct_table.copy()

    labels = apply_display_labels(
        plot_counts.index,
        label_map=label_map,
        wrap_width=wrap_width
    )

    n_cat = len(plot_counts)
    x = np.arange(n_cat)

    if figsize is None:
        width = max(10, min(18, n_cat * 1.5 + 4))
        height = 7.5 if n_cat <= 5 else 8.5
        figsize = (width, height)

    fig, ax = plt.subplots(figsize=figsize)

    bottom = np.zeros(n_cat)

    for size in size_order:
        if size not in plot_counts.columns:
            continue

        heights = plot_counts[size].fillna(0).values
        color = size_colors.get(size, "#D9D9D9")

        bars = ax.bar(
            x,
            heights,
            bottom=bottom,
            color=color,
            edgecolor="white",
            linewidth=1,
            width=0.64,
            label=size
        )

        # annotation ภายในแต่ละชั้น
        for i, bar in enumerate(bars):
            count_val = heights[i]

            if count_val <= 0:
                continue

            avg_budget = plot_budgets.loc[plot_counts.index[i], size] if size in plot_budgets.columns else np.nan

            # ถ้าชั้นเตี้ยมาก ให้ไม่ใส่เยอะเกินไป
            if count_val >= 2:
                label = f'{int(count_val)} โครงการ\n{avg_budget:.2f} ลบ.' if pd.notna(avg_budget) else f'{int(count_val)} โครงการ'
                ax.text(
                    bar.get_x() + bar.get_width()/2,
                    bottom[i] + count_val/2,
                    label,
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="#3A3A3A"
                )
            else:
                label = f'{int(count_val)}\n{avg_budget:.2f}' if pd.notna(avg_budget) else f'{int(count_val)}'
                ax.text(
                    bar.get_x() + bar.get_width()/2,
                    bottom[i] + count_val/2,
                    label,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="#3A3A3A"
                )

        bottom += heights

    # รวมบนยอดแท่ง
    max_total = max(bottom.max(), 1)

    for i, idx in enumerate(plot_counts.index):
        total_n = int(plot_totals.loc[idx, "จำนวน"])
        total_pct = plot_totals.loc[idx, "ร้อยละ"]

        ax.text(
            x[i],
            bottom[i] + max_total * 0.02,
            f"{total_n} ({total_pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

    ax.set_title(
        title,
        fontsize=17,
        fontweight="bold",
        pad=16
    )

    ax.set_ylabel(
        "จำนวนโครงการ",
        fontsize=13
    )

    ax.set_xlabel("")

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        fontsize=10
    )

    if rotate_xticks != 0:
        plt.setp(
            ax.get_xticklabels(),
            rotation=rotate_xticks,
            ha="right"
        )

    ax.grid(axis="y", linestyle="--", alpha=0.18)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    ax.set_ylim(0, max_total * 1.22)

    ax.legend(
        title="ขนาดโครงการ",
        frameon=False,
        fontsize=10,
        title_fontsize=10,
        loc="upper right"
    )

    if note:
        fig.text(
            0.01, 0.01,
            note,
            ha="left",
            va="bottom",
            fontsize=9,
            color="#555555"
        )
        fig.tight_layout(rect=[0, 0.04, 1, 1])
    else:
        fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=CHART_STYLE.get("save_dpi", 240),
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# PART 1
# ประเภทของการวิจัย
# ============================================================

research_type = frequency_table(
    df[research_type_col],
    denom=N
)
display(research_type[["จำนวน", "ร้อยละ", "n (%)"]])

type_budget = (
    df.groupby(research_type_col)
      .agg(
          n_projects=("รหัสโครงการ", "count"),
          budget_mean=(budget_col, "mean"),
          budget_total=(budget_col, "sum")
      )
      .rename(columns={
          "n_projects": "จำนวน",
          "budget_mean": "งบประมาณเฉลี่ย",
          "budget_total": "งบประมาณรวม"
      })
)

type_budget["ร้อยละ"] = type_budget["จำนวน"] / N * 100
type_budget["งบเฉลี่ย (ล้านบาท)"] = type_budget["งบประมาณเฉลี่ย"] / 1e6

display(
    type_budget[["จำนวน", "ร้อยละ", "งบเฉลี่ย (ล้านบาท)"]].round(2)
)

# cross-tab นับจำนวนโครงการ แยกตามขนาด
research_count_size = pd.crosstab(
    df[research_type_col],
    df[size_std_col]
).reindex(columns=size_order, fill_value=0)

# งบเฉลี่ย แยกตามขนาด
research_budget_size = (
    df.groupby([research_type_col, size_std_col])[budget_col]
      .mean()
      .unstack()
      .reindex(columns=size_order)
      / 1e6
)

# ให้เรียงตามลำดับใน research_type
research_count_size = research_count_size.reindex(research_type.index)
research_budget_size = research_budget_size.reindex(research_type.index)

# ตารางไขว้
cross_type_size = pd.crosstab(
    df[research_type_col].map(clean_text),
    df[size_std_col].map(clean_text),
    margins=True
)
display(cross_type_size)

# plot stacked bar
plot_stacked_count_with_budget(
    count_table=research_count_size,
    budget_table=research_budget_size,
    total_pct_table=research_type[["จำนวน", "ร้อยละ"]],
    title="ประเภทของการวิจัย",
    filename="4_3_2_research_type.png",
    wrap_width=24,
    figsize=(11, 7)
)


# ============================================================
# PART 2
# ประเภทของการวิจัยตามกรอบ บพท. (Multiple response)
# ============================================================

pmu_cols = [
    "ประเภทของการวิจัยตามกรอบ บพท. (1) (Gift)",
    "ประเภทของการวิจัยตามกรอบ (2) บพท. (Gift)",
    "ประเภทของการวิจัยตามกรอบ (3) (Gift)"
]

pmu_long = (
    df[["รหัสโครงการ", size_std_col, budget_col] + pmu_cols]
      .melt(
          id_vars=["รหัสโครงการ", size_std_col, budget_col],
          value_vars=pmu_cols,
          value_name="ประเภทตามกรอบ_บพท."
      )
      .dropna(subset=["ประเภทตามกรอบ_บพท."])
)

pmu_long["ประเภทตามกรอบ_บพท."] = pmu_long["ประเภทตามกรอบ_บพท."].map(clean_text)
pmu_long = pmu_long.drop_duplicates(["รหัสโครงการ", "ประเภทตามกรอบ_บพท."])

pmu_count = pmu_long["ประเภทตามกรอบ_บพท."].value_counts()
pmu_table = pd.DataFrame({"จำนวน": pmu_count})
pmu_table["ร้อยละ"] = pmu_table["จำนวน"] / N * 100
pmu_table["n (%)"] = [n_pct(n, N) for n in pmu_table["จำนวน"]]

display(pmu_table)

pmu_budget = (
    pmu_long.groupby("ประเภทตามกรอบ_บพท.")
      .agg(
          n_projects=("รหัสโครงการ", "nunique"),
          budget_mean=(budget_col, "mean")
      )
      .rename(columns={
          "n_projects": "จำนวน",
          "budget_mean": "งบประมาณเฉลี่ย"
      })
)

pmu_budget["ร้อยละโครงการ"] = pmu_budget["จำนวน"] / N * 100
pmu_budget["งบเฉลี่ย (ล้านบาท)"] = pmu_budget["งบประมาณเฉลี่ย"] / 1e6

display(
    pmu_budget[["จำนวน", "ร้อยละโครงการ", "งบเฉลี่ย (ล้านบาท)"]].round(2)
)

# count x size
pmu_count_size = pd.crosstab(
    pmu_long["ประเภทตามกรอบ_บพท."],
    pmu_long[size_std_col]
).reindex(columns=size_order, fill_value=0)

# budget mean x size
pmu_budget_size = (
    pmu_long.groupby(["ประเภทตามกรอบ_บพท.", size_std_col])[budget_col]
      .mean()
      .unstack()
      .reindex(columns=size_order)
      / 1e6
)

# เรียงตาม pmu_table
pmu_count_size = pmu_count_size.reindex(pmu_table.index)
pmu_budget_size = pmu_budget_size.reindex(pmu_table.index)

# plot stacked bar
plot_stacked_count_with_budget(
    count_table=pmu_count_size,
    budget_table=pmu_budget_size,
    total_pct_table=pmu_table[["จำนวน", "ร้อยละ"]],
    title="ประเภทของการวิจัยตามกรอบ บพท.",
    filename="4_3_2_pmu_research_framework.png",
    label_map=PMU_LABELS,
    wrap_width=20,
    figsize=(15, 8),
    rotate_xticks=25,
    note="หมายเหตุ: Multiple response — 1 โครงการอาจอยู่ได้มากกว่า 1 ประเภท"
)

# 4.3 การดำเนินงานของแผนงานวิจัย

## 4.3.1 หน่วยงานผู้รับทุน
ส่วนนี้ Word ระบุให้เขียนบรรยาย จึงใช้ตารางสังกัดจาก 4.2.3 เป็นข้อมูลประกอบ

## 4.3.2 ประเภทของงานวิจัย
วิเคราะห์ทั้งประเภทการวิจัยหลัก และประเภทการวิจัยตามกรอบ บพท. แบบ multiple response


In [ ]:
research_type_col = "ประเภทของการวิจัย"
research_type = frequency_table(df[research_type_col], denom=N)
display(research_type[["จำนวน", "ร้อยละ", "n (%)"]])

type_budget = (
    df.groupby(research_type_col)
      .agg(
          n_projects=("รหัสโครงการ", "count"),
          budget_mean=("งบประมาณที่ได้รับจัดสรร", "mean"),
          budget_total=("งบประมาณที่ได้รับจัดสรร", "sum")
      )
      .rename(columns={"n_projects":"จำนวน", "budget_mean":"งบประมาณเฉลี่ย", "budget_total":"งบประมาณรวม"})
)
type_budget["ร้อยละ"] = type_budget["จำนวน"]/N*100
type_budget["งบเฉลี่ย (ล้านบาท)"] = type_budget["งบประมาณเฉลี่ย"]/1e6
display(type_budget[["จำนวน", "ร้อยละ", "งบเฉลี่ย (ล้านบาท)"]].round(2))

# ============================================================
# กราฟ 4.3.2 (1) ประเภทของการวิจัย
# Bar = จำนวนโครงการ
# จุด = งบประมาณเฉลี่ย แยกตามขนาด เล็ก / กลาง / ใหญ่
# ============================================================

plot_df = research_type[["จำนวน", "ร้อยละ"]].copy()

budget_by_size = (
    df.groupby(
        [research_type_col, "ขนาดโครงการ"]
    )["งบประมาณที่ได้รับจัดสรร"]
    .mean()
    .unstack()
    / 1e6
)

budget_by_size = budget_by_size.reindex(plot_df.index)

x = np.arange(len(plot_df))

fig, ax1 = plt.subplots(figsize=(11, 7))

bar_colors = [
    "#8FB9E0",
    "#A8D5BA",
    "#F6C28B"
]

bars = ax1.bar(
    x,
    plot_df["จำนวน"].values,
    color=bar_colors[:len(plot_df)],
    edgecolor="white",
    linewidth=1,
    width=0.62
)

ax1.set_title(
    "ประเภทของการวิจัย",
    fontsize=17,
    fontweight="bold",
    pad=16
)

ax1.set_ylabel("จำนวนโครงการ", fontsize=13)
ax1.set_xlabel("")

ax1.set_xticks(x)
ax1.set_xticklabels(
    plot_df.index,
    fontsize=11
)

ax1.grid(
    axis="y",
    linestyle="--",
    alpha=0.18
)

ax1.set_axisbelow(True)
ax1.spines[["top"]].set_visible(False)

max_count = max(plot_df["จำนวน"].max(), 1)

for bar, (_, row) in zip(
    bars,
    plot_df.iterrows()
):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + max_count*0.02,
        f'{int(row["จำนวน"])} ({row["ร้อยละ"]:.1f}%)',
        ha="center",
        va="bottom",
        fontsize=11
    )

ax1.set_ylim(
    0,
    max_count * 1.22
)

# ------------------------------------------------------------
# แกน Y ขวา = งบประมาณเฉลี่ย
# ------------------------------------------------------------

ax2 = ax1.twinx()

ax2.set_ylabel(
    "งบประมาณเฉลี่ย (ล้านบาท/โครงการ)",
    fontsize=13
)

offsets = {
    "เล็ก": -0.16,
    "กลาง": 0,
    "ใหญ่": 0.16
}

markers = {
    "เล็ก": "o",
    "กลาง": "s",
    "ใหญ่": "^"
}

marker_colors = {
    "เล็ก": "#6BAED6",
    "กลาง": "#74C476",
    "ใหญ่": "#FD8D3C"
}

for size in ["เล็ก", "กลาง", "ใหญ่"]:

    if size not in budget_by_size.columns:
        continue

    values = budget_by_size[size]
    valid = values.notna()

    xx = x[valid] + offsets[size]
    yy = values[valid].values

    ax2.scatter(
        xx,
        yy,
        s=90,
        marker=markers[size],
        color=marker_colors[size],
        edgecolor="white",
        linewidth=1,
        label=f"ขนาด{size}",
        zorder=5
    )

    for xpos, val in zip(xx, yy):
        ax2.annotate(
            f"{val:.2f}",
            (xpos, val),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            color=marker_colors[size]
        )

ax2.legend(
    loc="upper right",
    frameon=False,
    fontsize=10,
    title="งบเฉลี่ย"
)

ax2.spines["top"].set_visible(False)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "4_3_2_research_type.png",
    dpi=CHART_STYLE.get("save_dpi", 240),
    bbox_inches="tight"
)

plt.show()

# ประเภทการวิจัย x ขนาดโครงการ
cross_type_size = pd.crosstab(
    df[research_type_col].map(clean_text),
    df["ขนาดโครงการ"].map(clean_text),
    margins=True
)
display(cross_type_size)

# ประเภทงานวิจัยตามกรอบ บพท. เป็น multiple response
pmu_cols = [
    "ประเภทของการวิจัยตามกรอบ บพท. (1) (Gift)",
    "ประเภทของการวิจัยตามกรอบ (2) บพท. (Gift)",
    "ประเภทของการวิจัยตามกรอบ (3) (Gift)"
]
pmu_long = (
    df[["รหัสโครงการ", "ขนาดโครงการ", "งบประมาณที่ได้รับจัดสรร"] + pmu_cols]
      .melt(
          id_vars=["รหัสโครงการ", "ขนาดโครงการ", "งบประมาณที่ได้รับจัดสรร"],
          value_vars=pmu_cols,
          value_name="ประเภทตามกรอบ_บพท."
      )
      .dropna(subset=["ประเภทตามกรอบ_บพท."])
)
pmu_long["ประเภทตามกรอบ_บพท."] = pmu_long["ประเภทตามกรอบ_บพท."].map(clean_text)
pmu_long = pmu_long.drop_duplicates(["รหัสโครงการ", "ประเภทตามกรอบ_บพท."])

pmu_count = pmu_long["ประเภทตามกรอบ_บพท."].value_counts()
pmu_table = pd.DataFrame({"จำนวน": pmu_count})
pmu_table["ร้อยละ"] = pmu_table["จำนวน"]/N*100
pmu_table["n (%)"] = [n_pct(n, N) for n in pmu_table["จำนวน"]]
display(pmu_table)

# ============================================================
# กราฟ 4.3.2 (2) ประเภทของการวิจัยตามกรอบ บพท.
# Bar = จำนวนโครงการ
# จุด = งบประมาณเฉลี่ย แยกตามขนาด เล็ก / กลาง / ใหญ่
# ============================================================

plot_df = pmu_table[["จำนวน", "ร้อยละ"]].copy()

pmu_budget_size = (
    pmu_long.groupby(
        ["ประเภทตามกรอบ_บพท.", "ขนาดโครงการ"]
    )["งบประมาณที่ได้รับจัดสรร"]
    .mean()
    .unstack()
    / 1e6
)

pmu_budget_size = pmu_budget_size.reindex(plot_df.index)

display_labels = apply_display_labels(
    plot_df.index,
    label_map=PMU_LABELS,
    wrap_width=22
)

x = np.arange(len(plot_df))

fig, ax1 = plt.subplots(figsize=(14, 8))

bar_colors = [
    "#8FB9E0",
    "#A8D5BA",
    "#F6C28B",
    "#C7B6E5",
    "#F2B5B5",
    "#9ED9CC",
    "#F7D6A3"
]

bars = ax1.bar(
    x,
    plot_df["จำนวน"].values,
    color=bar_colors[:len(plot_df)],
    edgecolor="white",
    linewidth=1,
    width=0.62
)

ax1.set_title(
    "ประเภทของการวิจัยตามกรอบ บพท.",
    fontsize=17,
    fontweight="bold",
    pad=16
)

ax1.set_ylabel("จำนวนโครงการ", fontsize=13)
ax1.set_xlabel("")

ax1.set_xticks(x)
ax1.set_xticklabels(
    display_labels,
    fontsize=10,
    rotation=25,
    ha="right"
)

ax1.grid(
    axis="y",
    linestyle="--",
    alpha=0.18
)

ax1.set_axisbelow(True)
ax1.spines[["top"]].set_visible(False)

max_count = max(plot_df["จำนวน"].max(), 1)

for bar, (_, row) in zip(
    bars,
    plot_df.iterrows()
):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + max_count*0.02,
        f'{int(row["จำนวน"])} ({row["ร้อยละ"]:.1f}%)',
        ha="center",
        va="bottom",
        fontsize=10
    )

ax1.set_ylim(
    0,
    max_count * 1.25
)

# ------------------------------------------------------------
# แกน Y ขวา = งบประมาณเฉลี่ย
# ------------------------------------------------------------

ax2 = ax1.twinx()

ax2.set_ylabel(
    "งบประมาณเฉลี่ย (ล้านบาท/โครงการ)",
    fontsize=13
)

offsets = {
    "เล็ก": -0.16,
    "กลาง": 0,
    "ใหญ่": 0.16
}

markers = {
    "เล็ก": "o",
    "กลาง": "s",
    "ใหญ่": "^"
}

marker_colors = {
    "เล็ก": "#6BAED6",
    "กลาง": "#74C476",
    "ใหญ่": "#FD8D3C"
}

for size in ["เล็ก", "กลาง", "ใหญ่"]:

    if size not in pmu_budget_size.columns:
        continue

    values = pmu_budget_size[size]
    valid = values.notna()

    xx = x[valid] + offsets[size]
    yy = values[valid].values

    ax2.scatter(
        xx,
        yy,
        s=90,
        marker=markers[size],
        color=marker_colors[size],
        edgecolor="white",
        linewidth=1,
        label=f"ขนาด{size}",
        zorder=5
    )

    for xpos, val in zip(xx, yy):
        ax2.annotate(
            f"{val:.2f}",
            (xpos, val),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            color=marker_colors[size]
        )

ax2.legend(
    loc="upper right",
    frameon=False,
    fontsize=10,
    title="งบเฉลี่ย"
)

ax2.spines["top"].set_visible(False)

fig.text(
    0.01,
    0.01,
    "หมายเหตุ: Multiple response — 1 โครงการอาจอยู่ได้มากกว่า 1 ประเภท",
    fontsize=9
)

fig.tight_layout(
    rect=[0, 0.04, 1, 1]
)

fig.savefig(
    OUTPUT_DIR / "4_3_2_pmu_research_framework.png",
    dpi=CHART_STYLE.get("save_dpi", 240),
    bbox_inches="tight"
)

plt.show()

pmu_budget = (
    pmu_long.groupby("ประเภทตามกรอบ_บพท.")
      .agg(
          n_projects=("รหัสโครงการ", "nunique"),
          budget_mean=("งบประมาณที่ได้รับจัดสรร", "mean")
      )
      .rename(columns={"n_projects":"จำนวน", "budget_mean":"งบประมาณเฉลี่ย"})
)
pmu_budget["ร้อยละโครงการ"] = pmu_budget["จำนวน"]/N*100
pmu_budget["งบเฉลี่ย (ล้านบาท)"] = pmu_budget["งบประมาณเฉลี่ย"]/1e6
display(pmu_budget[["จำนวน", "ร้อยละโครงการ", "งบเฉลี่ย (ล้านบาท)"]].round(2))


## 4.3.3 สาขางานวิจัย (OECD)

ใช้ `สาขาการวิจัย (OECD1)` เป็นสาขาหลักตามกรอบ Word และแสดงจำนวน/ร้อยละกำกับทุกแท่ง


In [ ]:
oecd_field = frequency_table(df["สาขาการวิจัย (OECD1)"], denom=N)
display(oecd_field[["จำนวน", "ร้อยละ", "n (%)"]])
save_table(oecd_field, "4_3_3_oecd_research_field.csv")

barh_count(
    oecd_field[["จำนวน", "ร้อยละ"]],
    "สาขาการวิจัยหลักของโครงการ (OECD)",
    filename="4_3_3_oecd_research_field.png",
    label_map=OECD_FIELD_LABELS,
    wrap_width=28,
    color=COLORS["primary"]
)


# 4.4 ผลผลิตของแผนงานวิจัย

## 4.4.1 ประเภทของผลผลิต

หนึ่งโครงการอาจมีผลผลิตได้หลายประเภท จึงแปลงผลผลิตลำดับที่ 1–6 เป็น long format ก่อนนับ  
**ข้อควรระวัง:** หากนำงบประมาณโครงการไปผูกกับผลผลิตหลายประเภท งบประมาณจะถูกนับซ้ำเมื่อรวมข้ามประเภท ดังนั้นตารางนี้เหมาะสำหรับเปรียบเทียบ “งบของโครงการที่มีผลผลิตประเภทนั้น” ไม่ควรนำยอดข้ามประเภทมาบวกเป็นงบรวมแผนงาน


In [ ]:
output_cols = [
    "ผลผลิตหลัก ลำดับที่ 1",
    "ผลผลิตหลัก ลำดับที่ 2",
    "ผลผลิตหลัก ลำดับที่ 3",
    "ผลผลิตหลัก ลำดับที่ 4",
    "ผลผลิตหลัก ลำดับที่ 5",
    "ผลผลิตหลัก ลำดับที่ 6",
]

output_long = (
    df[["รหัสโครงการ", research_type_col, "งบประมาณที่ได้รับจัดสรร"] + output_cols]
      .melt(
          id_vars=["รหัสโครงการ", research_type_col, "งบประมาณที่ได้รับจัดสรร"],
          value_vars=output_cols,
          value_name="ประเภทผลผลิต"
      )
      .dropna(subset=["ประเภทผลผลิต"])
)
output_long["ประเภทผลผลิต"] = output_long["ประเภทผลผลิต"].map(clean_text)
output_long = output_long.drop_duplicates(["รหัสโครงการ", "ประเภทผลผลิต"])

output_count = output_long["ประเภทผลผลิต"].value_counts()
output_table = pd.DataFrame({"จำนวน": output_count})
output_table["ร้อยละ"] = output_table["จำนวน"]/N*100
output_table["n (%)"] = [n_pct(n, N) for n in output_table["จำนวน"]]

display(output_table)
save_table(output_table, "4_4_1_output_types.csv")

barh_count(
    output_table[["จำนวน", "ร้อยละ"]],
    "ประเภทผลผลิตของโครงการ (Multiple response)",
    filename="4_4_1_output_types.png",
    label_map=OUTPUT_LABELS,
    wrap_width=30,
    color=COLORS["primary"],
    figsize=(11, max(6, 0.62*len(output_table)+2)),
    note="หมายเหตุ: Multiple response — 1 โครงการอาจมีผลผลิตมากกว่า 1 ประเภท"
)

# ผลผลิต x ประเภทการวิจัย
output_by_type = pd.crosstab(
    output_long["ประเภทผลผลิต"],
    output_long[research_type_col]
)
display(output_by_type)

# งบประมาณของโครงการที่มีผลผลิตแต่ละประเภท
output_budget = (
    output_long.groupby("ประเภทผลผลิต")
      .agg(
          n_projects=("รหัสโครงการ", "nunique"),
          budget_total=("งบประมาณที่ได้รับจัดสรร", "sum"),
          budget_mean=("งบประมาณที่ได้รับจัดสรร", "mean")
      )
      .rename(columns={
          "n_projects":"จำนวนโครงการ",
          "budget_total":"งบประมาณรวมของโครงการ",
          "budget_mean":"งบประมาณเฉลี่ยต่อโครงการ"
      })
)
output_budget["งบรวมของโครงการ (ล้านบาท)"] = output_budget["งบประมาณรวมของโครงการ"]/1e6
output_budget["งบเฉลี่ยต่อโครงการ (ล้านบาท)"] = output_budget["งบประมาณเฉลี่ยต่อโครงการ"]/1e6
display(
    output_budget[
        ["จำนวนโครงการ", "งบรวมของโครงการ (ล้านบาท)", "งบเฉลี่ยต่อโครงการ (ล้านบาท)"]
    ].round(2)
)


## 4.4.2 TRL ณ สิ้นสุดโครงการ

กราฟใช้ระดับ TRL เรียงจากต่ำไปสูง ไม่เรียงตามจำนวน เพื่อให้เห็น “ระดับความพร้อม” เป็นลำดับต่อเนื่อง


In [ ]:
trl = numeric_prefix(df["TRL สิ้นสุด (Gift)"])
trl_label = trl.map(lambda x: f"TRL {int(x)}" if pd.notna(x) else "N/A")
trl_table = frequency_table(trl_label, denom=N, sort=False)

trl_order = [f"TRL {i}" for i in range(1,10)] + ["N/A"]
trl_table = trl_table.reindex([x for x in trl_order if x in trl_table.index])

# เพิ่มงบ
trl_budget = (
    df.assign(TRL_label=trl_label)
      .groupby("TRL_label", dropna=False)["งบประมาณที่ได้รับจัดสรร"]
      .sum()
)
trl_table["งบประมาณรวม"] = trl_budget.reindex(trl_table.index)
trl_table["สัดส่วนงบประมาณ"] = trl_table["งบประมาณรวม"]/budget.sum()*100
trl_table["งบรวม (ล้านบาท)"] = trl_table["งบประมาณรวม"]/1e6

display(trl_table[["จำนวน", "ร้อยละ", "งบรวม (ล้านบาท)", "สัดส่วนงบประมาณ"]].round(2))
save_table(trl_table, "4_4_2_trl.csv")

barh_count(
    trl_table[["จำนวน", "ร้อยละ"]],
    "ระดับความพร้อมทางเทคโนโลยี (TRL) ณ สิ้นสุดโครงการ",
    filename="4_4_2_trl.png",
    color=COLORS["purple"],
    wrap_width=18,
    sort_by_count=False
)


## 4.4.3 SRL ณ สิ้นสุดโครงการ

ใช้หลักเดียวกับ TRL โดยเรียงระดับ SRL จากต่ำไปสูง และคง N/A แยกต่างหาก


In [ ]:
srl = numeric_prefix(df["SRL สิ้นสุด(Gift)"])
srl_label = srl.map(lambda x: f"SRL {int(x)}" if pd.notna(x) else "N/A")
srl_table = frequency_table(srl_label, denom=N, sort=False)

srl_order = [f"SRL {i}" for i in range(1,10)] + ["N/A"]
srl_table = srl_table.reindex([x for x in srl_order if x in srl_table.index])

srl_budget = (
    df.assign(SRL_label=srl_label)
      .groupby("SRL_label", dropna=False)["งบประมาณที่ได้รับจัดสรร"]
      .sum()
)
srl_table["งบประมาณรวม"] = srl_budget.reindex(srl_table.index)
srl_table["สัดส่วนงบประมาณ"] = srl_table["งบประมาณรวม"]/budget.sum()*100
srl_table["งบรวม (ล้านบาท)"] = srl_table["งบประมาณรวม"]/1e6

display(srl_table[["จำนวน", "ร้อยละ", "งบรวม (ล้านบาท)", "สัดส่วนงบประมาณ"]].round(2))
save_table(srl_table, "4_4_3_srl.csv")

barh_count(
    srl_table[["จำนวน", "ร้อยละ"]],
    "ระดับความพร้อมของความรู้และเทคโนโลยีทางด้านสังคม (SRL) ณ สิ้นสุดโครงการ",
    filename="4_4_3_srl.png",
    color=COLORS["purple"],
    wrap_width=18,
    sort_by_count=False
)


# 4.5 ผลประโยชน์ทางวิชาการ

## 4.5.1 ผลประโยชน์ระดับผลผลิต

รวมจำนวนบทความ การเผยแพร่ ตำรา/คู่มือ สื่อ และการใช้ประโยชน์ด้านการเรียนการสอนตามคอลัมน์ในฐานข้อมูล  
ส่วน “การพัฒนาบุคลากร” มีเพียงตัวแปรที่ฐานข้อมูลรองรับ จึงไม่เติมตัวเลขจากแหล่งอื่น


In [ ]:
academic_cols = [
    "จำนวนบทความ (ระดับนานาชาติ)",
    "จำนวนร่างบทความ (ระดับนานาชาติ)",
    "จำนวนบทความ (ระดับประเทศ)",
    "จำนวนร่างบทความ (ระดับประเทศ)",
    "จำนวนบทความ (นำเสนอในที่ประชุมระดับนานาชาติ)",
    "จำนวนบทความ (นำเสนอในที่ประชุมระดับประเทศ)",
    "จำนวน ตำรา/หนังสือ (จำนวนเรื่อง)",
    "จำนวนคู่มือ/สิ่งพิมพ์",
    "จำนวนครั้งการเผยแพร่ผ่านการอบรม/สัมมนา/เวทีสาธารณะ/นิทรรศการ",
    "จำนวนสื่อ clip vdo หรือเพจเผยแพร่/งานเขียนออนไลน์",
    "จำนวนครั้งการเผยแพร่ผ่าน วิดีทัศน์ โทรทัศน์ วิทยุ นสพ. อินเตอร์เน็ต",
    "มีการใช้ประโยชน์กับการเรียนการสอน (จำนวนวิชา)",
    "มีการใช้ประโยชน์กับการวิจัยเพื่อพัฒนานิสิต (จำนวนนิสิต ที่ทำวิจัยหรือวิทยานิพนธ์)",
]

acad_sum = {}
for c in academic_cols:
    acad_sum[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).sum()

academic_table = (
    pd.Series(acad_sum, name="จำนวนรวม")
      .sort_values(ascending=False)
      .to_frame()
)
display(academic_table)
save_table(academic_table, "4_5_1_academic_benefits.csv")

plot_acad = academic_table[academic_table["จำนวนรวม"] > 0].copy()
if not plot_acad.empty:
    plot_acad = plot_acad.sort_values("จำนวนรวม", ascending=False)
    display_labels = apply_display_labels(plot_acad.index, label_map=ACADEMIC_LABELS, wrap_width=24)
    n_cat = len(plot_acad)
    fig, ax = plt.subplots(figsize=(max(12, n_cat * 1.15), 8.2))
    bars = ax.bar(
        range(n_cat),
        plot_acad["จำนวนรวม"].values,
        color=COLORS["academic"],
        edgecolor="white",
        linewidth=0.8,
    )
    ax.set_title("ผลประโยชน์ทางวิชาการระดับผลผลิต", weight="bold", pad=14, fontsize=CHART_STYLE["title_size"])
    ax.set_ylabel("จำนวนรวม", fontsize=CHART_STYLE["axis_label_size"])
    ax.set_xlabel("")
    ax.set_xticks(range(n_cat))
    ax.set_xticklabels(display_labels, fontsize=CHART_STYLE["tick_size"])
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.tick_params(axis="y", labelsize=CHART_STYLE["tick_size"])
    ax.grid(axis="y", alpha=CHART_STYLE["grid_alpha"], linestyle="--")
    ax.set_axisbelow(True)
    ax.spines[["top","right"]].set_visible(False)
    maxv = max(plot_acad["จำนวนรวม"].max(), 1)
    ax.set_ylim(0, maxv*1.18 + 1)
    for rect, value in zip(bars, plot_acad["จำนวนรวม"].values):
        ax.annotate(
            f"{int(value)}",
            xy=(rect.get_x()+rect.get_width()/2, rect.get_height()),
            xytext=(0, 5), textcoords="offset points",
            ha="center", va="bottom",
            fontsize=CHART_STYLE["annotation_size"], color=COLORS["dark"]
        )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/"4_5_1_academic_benefits.png", dpi=CHART_STYLE["save_dpi"], bbox_inches="tight")
    plt.show()

# ทรัพย์สินทางปัญญา
patent_status = pd.to_numeric(
    df["สิทธิบัตร สถานะ 0= ไม่มี, 1=ได้แล้ว2=รออนุมัติ 3=กำลังจะจด"],
    errors="coerce"
)
petty_patent_status = pd.to_numeric(
    df["อนุสิทธิบัตร สถานะ 0= ไม่มี, 1=ได้แล้ว2=รออนุมัติ 3=กำลังจะจด"],
    errors="coerce"
)

ip_summary = pd.DataFrame({
    "สิทธิบัตร": patent_status.value_counts().sort_index(),
    "อนุสิทธิบัตร": petty_patent_status.value_counts().sort_index(),
}).fillna(0).astype(int)

ip_summary.index = [
    {0:"0 = ไม่มี", 1:"1 = ได้แล้ว", 2:"2 = รออนุมัติ", 3:"3 = กำลังจะจด"}.get(int(i), str(i))
    for i in ip_summary.index
]
display(ip_summary)



## 4.5.2 ผลประโยชน์ทางวิชาการระดับผลลัพธ์

เอกสาร Word ต้องการ “การได้รับอ้างอิงบทความทางวิชาการระดับนานาชาติ” แต่ในชีต `สถานภาพแผนงาน -Clean` **ไม่พบตัวแปรจำนวนการอ้างอิงโดยตรง**  
จึงไม่คำนวณแทนด้วยตัวแปรอื่น เพื่อไม่ให้ความหมายคลาดเคลื่อน


# 4.6 ผลลัพธ์ (Outcome) และผลกระทบ (Impacts)

## 4.6.1.1 กลุ่มผู้ใช้ประโยชน์

ผู้ใช้ประโยชน์เป็น multiple response โดยรวมคอลัมน์ผู้ใช้ประโยชน์ที่ 1–5  
จากนั้นวิเคราะห์ซ้ำเฉพาะโครงการที่ฐานข้อมูลระบุว่า “เกิดผลลัพธ์แล้ว”


In [ ]:
user_cols = [f"ผู้ใช้ประโยชน์ที่ {i}" for i in range(1,6)]

user_long = (
    df[["รหัสโครงการ", "เกิดผลลัพธ์ แล้วหรือไม่"] + user_cols]
      .melt(
          id_vars=["รหัสโครงการ", "เกิดผลลัพธ์ แล้วหรือไม่"],
          value_vars=user_cols,
          value_name="กลุ่มผู้ใช้ประโยชน์"
      )
      .dropna(subset=["กลุ่มผู้ใช้ประโยชน์"])
)
user_long["กลุ่มผู้ใช้ประโยชน์"] = user_long["กลุ่มผู้ใช้ประโยชน์"].map(clean_text)
user_long = user_long.drop_duplicates(["รหัสโครงการ", "กลุ่มผู้ใช้ประโยชน์"])

user_count = user_long["กลุ่มผู้ใช้ประโยชน์"].value_counts()
user_table = pd.DataFrame({"จำนวน": user_count})
user_table["ร้อยละ"] = user_table["จำนวน"]/N*100
user_table["n (%)"] = [n_pct(n, N) for n in user_table["จำนวน"]]

display(user_table)
barh_count(
    user_table[["จำนวน","ร้อยละ"]],
    "กลุ่มผู้ใช้ประโยชน์เป้าหมาย (Multiple response)",
    filename="4_6_1_1_target_users.png",
    label_map=USER_LABELS,
    wrap_width=28,
    color=COLORS["secondary"],
    note="หมายเหตุ: Multiple response — 1 โครงการอาจมีกลุ่มผู้ใช้มากกว่า 1 กลุ่ม"
)

# เกิดผลลัพธ์แล้ว: รองรับทั้งเลข 1 และข้อความที่ขึ้นต้นด้วย 1
outcome_happened = numeric_prefix(df["เกิดผลลัพธ์ แล้วหรือไม่"]).eq(1)
actual_outcome_ids = set(df.loc[outcome_happened, "รหัสโครงการ"])

actual_user_long = user_long[user_long["รหัสโครงการ"].isin(actual_outcome_ids)]
actual_user_count = actual_user_long["กลุ่มผู้ใช้ประโยชน์"].value_counts()

actual_user_table = pd.DataFrame({"จำนวน": actual_user_count})
actual_denom = len(actual_outcome_ids)
actual_user_table["ร้อยละ"] = actual_user_table["จำนวน"]/actual_denom*100 if actual_denom else 0
actual_user_table["n (%)"] = [n_pct(n, actual_denom) for n in actual_user_table["จำนวน"]]

print("จำนวนโครงการที่ระบุว่าเกิดผลลัพธ์แล้ว:", actual_denom)
display(actual_user_table)

if not actual_user_table.empty:
    barh_count(
        actual_user_table[["จำนวน","ร้อยละ"]],
        "กลุ่มผู้ใช้ประโยชน์ในโครงการที่ระบุว่าเกิดผลลัพธ์แล้ว",
        filename="4_6_1_1_actual_users.png",
        label_map=USER_LABELS,
        wrap_width=28,
        color=COLORS["secondary"],
        note="หมายเหตุ: Multiple response — คำนวณจากโครงการที่ระบุว่าเกิดผลลัพธ์แล้ว"
    )


## 4.6.1.2 ประเภทผลลัพธ์ของงานวิจัย

รวมผลลัพธ์ที่ 1–5 เป็น multiple response และสามารถ cross-tab ตามขนาดโครงการ ประเภทการวิจัย และกลุ่มผู้ใช้ประโยชน์ได้


In [ ]:
outcome_cols = [f"ผลลัพธ์ที่ {i}" for i in range(1,6)]

outcome_long = (
    df[["รหัสโครงการ", "ขนาดโครงการ", research_type_col] + outcome_cols]
      .melt(
          id_vars=["รหัสโครงการ", "ขนาดโครงการ", research_type_col],
          value_vars=outcome_cols,
          value_name="ประเภทผลลัพธ์"
      )
      .dropna(subset=["ประเภทผลลัพธ์"])
)
outcome_long["ประเภทผลลัพธ์"] = outcome_long["ประเภทผลลัพธ์"].map(clean_text)
outcome_long = outcome_long.drop_duplicates(["รหัสโครงการ", "ประเภทผลลัพธ์"])

outcome_count = outcome_long["ประเภทผลลัพธ์"].value_counts()
outcome_table = pd.DataFrame({"จำนวน": outcome_count})
outcome_table["ร้อยละ"] = outcome_table["จำนวน"]/N*100
outcome_table["n (%)"] = [n_pct(n, N) for n in outcome_table["จำนวน"]]

display(outcome_table)
save_table(outcome_table, "4_6_1_2_outcome_types.csv")

barh_count(
    outcome_table[["จำนวน","ร้อยละ"]],
    "ประเภทผลลัพธ์ของงานวิจัย (Multiple response)",
    filename="4_6_1_2_outcome_types.png",
    label_map=OUTCOME_LABELS,
    wrap_width=32,
    color=COLORS["secondary"],
    figsize=(11, max(6, 0.62*len(outcome_table)+2)),
    note="หมายเหตุ: Multiple response — 1 โครงการอาจมีผลลัพธ์มากกว่า 1 ประเภท"
)

print("ผลลัพธ์ x ขนาดโครงการ")
display(pd.crosstab(outcome_long["ประเภทผลลัพธ์"], outcome_long["ขนาดโครงการ"]))

print("ผลลัพธ์ x ประเภทการวิจัย")
display(pd.crosstab(outcome_long["ประเภทผลลัพธ์"], outcome_long[research_type_col]))


## 4.6.2 ผลกระทบ

ฐานข้อมูลระบุ `0/1` ว่าโครงการมีผลกระทบ **ที่เกิดขึ้นหรือคาดว่าจะเกิดขึ้น** ในมิติเศรษฐกิจ สังคม และสิ่งแวดล้อม  
กราฟนี้จึงควรเขียนคำกำกับว่า **“เกิดขึ้น/คาดว่าจะเกิดขึ้น”** และไม่ตีความทุกค่า 1 ว่าเป็น Actual Impact


In [ ]:
impact_cols = {
    "เศรษฐกิจ": "ด้านเศรษฐกิจ",
    "สังคม": "ด้านสังคม",
    "สิ่งแวดล้อม": "ด้านสิ่งแวดล้อม"
}

impact_rows = []
for label, col in impact_cols.items():
    vals = pd.to_numeric(df[col], errors="coerce")
    n = vals.eq(1).sum()
    impact_rows.append([label, n, n/N*100])

impact_table = pd.DataFrame(
    impact_rows,
    columns=["มิติผลกระทบ", "จำนวน", "ร้อยละ"]
).set_index("มิติผลกระทบ")
impact_table["n (%)"] = [n_pct(n, N) for n in impact_table["จำนวน"]]

display(impact_table)
save_table(impact_table, "4_6_2_impact_dimensions.csv")

barh_count(
    impact_table[["จำนวน","ร้อยละ"]],
    "มิติผลกระทบที่เกิดขึ้น/คาดว่าจะเกิดขึ้น",
    filename="4_6_2_impact_dimensions.png",
    color=COLORS["accent"],
    wrap_width=24
)

# จำแนกผลกระทบตามขนาดโครงการและประเภทการวิจัย
for label, col in impact_cols.items():
    temp = df.loc[pd.to_numeric(df[col], errors="coerce").eq(1)]
    print(f"\n{label} — จำแนกตามขนาดโครงการ")
    display(temp["ขนาดโครงการ"].value_counts().to_frame("จำนวน"))
    print(f"{label} — จำแนกตามประเภทการวิจัย")
    display(temp[research_type_col].value_counts().to_frame("จำนวน"))


# 4.7 การประเมินผลสำเร็จตามเกณฑ์ OECD

วิเคราะห์ 6 มิติ:
1. Relevance
2. Coherence
3. Effectiveness
4. Efficiency
5. Outcomes & Impact
6. Sustainability

สำหรับกราฟใยแมงมุมด้านล่าง:
- 4 มิติแรกใช้ **สัดส่วนโครงการที่ได้ค่า 1**
- Outcomes & Impact ใช้ค่า 0–3 แล้วหาร 3 เพื่อ normalize เป็น 0–1 และ **ตัดรหัส 4 = N/A ออก**
- Sustainability ใช้ระดับ 1–3 โดย **1 = ยั่งยืนกว่า, 3 = ไม่ยั่งยืนกว่า** จึงแปลงเป็น `(3-score)/2`; รหัส 4 = N/A ถูกตัดออก
- กราฟนี้เป็น **descriptive normalized summary** เพื่อสื่อสารภาพรวม ไม่ใช่คะแนน OECD มาตรฐานอย่างเป็นทางการ


In [ ]:
binary_oecd = {
    "Relevance": "Relevance (0-ไม่มี, 1 -มี)",
    "Coherence": "Coherence (0-ไม่มี, 1 -มี)",
    "Effectiveness": "Effectiveness (0-ไม่มี, 1 -มี)",
    "Efficiency": "Efficiency (0-ไม่มี, 1 -มี)",
}

oecd_summary_rows = []
for label, col in binary_oecd.items():
    vals = pd.to_numeric(df[col], errors="coerce")
    n1 = vals.eq(1).sum()
    valid = vals.notna().sum()
    oecd_summary_rows.append([label, n1, valid, n1/valid*100 if valid else np.nan])

oecd_binary_table = pd.DataFrame(
    oecd_summary_rows,
    columns=["เกณฑ์", "จำนวนผ่าน", "จำนวนที่มีข้อมูล", "ร้อยละผ่าน"]
).set_index("เกณฑ์")
display(oecd_binary_table.round(2))

# Distribution: Outcome & Impact
oi_score = numeric_prefix(df["Outcomes and Impact (Dropdownlist)"])
oi_table = frequency_table(oi_score, denom=N, dropna=False, sort=False)
display(Markdown("### Outcomes & Impact"))
display(oi_table)

# Distribution: Sustainability
sus_score = numeric_prefix(df["Sustainability (Dropdownlist)"])
sus_table = frequency_table(sus_score, denom=N, dropna=False, sort=False)
display(Markdown("### Sustainability"))
display(sus_table)

# Radar normalized summary
radar_labels = ["Relevance", "Coherence", "Effectiveness", "Efficiency", "Outcome & Impact", "Sustainability"]

radar_values = []
for label in radar_labels[:4]:
    radar_values.append(oecd_binary_table.loc[label, "ร้อยละผ่าน"]/100)

oi_valid = oi_score[oi_score.isin([0,1,2,3])]
radar_values.append((oi_valid/3).mean() if len(oi_valid) else np.nan)

sus_valid = sus_score[sus_score.isin([1,2,3])]
radar_values.append(((3-sus_valid)/2).mean() if len(sus_valid) else np.nan)

radar_df = pd.DataFrame({"มิติ": radar_labels, "ค่าปรับมาตรฐาน 0–1": radar_values})
display(radar_df.round(3))

angles = np.linspace(0, 2*np.pi, len(radar_labels), endpoint=False).tolist()
values = radar_values + radar_values[:1]
angles_closed = angles + angles[:1]

fig = plt.figure(figsize=(7.5, 7.5))
ax = fig.add_subplot(111, polar=True)
ax.plot(angles_closed, values, linewidth=2, marker="o")
ax.fill(angles_closed, values, alpha=0.10)
ax.set_xticks(angles)
ax.set_xticklabels(radar_labels)
ax.set_ylim(0, 1)
ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"])
ax.set_title("ภาพรวมผลสำเร็จตามเกณฑ์ OECD\n(Descriptive normalized summary)", pad=24, weight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"4_7_oecd_radar.png", bbox_inches="tight")
plt.show()


## ตารางสรุปสำหรับนำไปเขียนบทที่ 4

Cell ด้านล่างสร้างตารางสรุปตัวเลขหลักแบบสั้น เพื่อคัดลอกไปใช้เขียน narrative ได้ทันที


In [ ]:
summary_rows = [
    ["จำนวนโครงการทั้งหมด", N, "100.0%"],
    ["งบประมาณรวม (ล้านบาท)", round(budget.sum()/1e6, 2), ""],
    ["งบประมาณเฉลี่ยต่อโครงการ (ล้านบาท)", round(budget.mean()/1e6, 2), ""],
    ["โครงการขยายเวลา", int(extended.sum()), f"{extended.mean()*100:.1f}%"],
    ["โครงการยุติ", int(terminated.sum()), f"{terminated.mean()*100:.1f}%"],
    ["นักวิจัยรวม (คน)", int(researcher_n.sum()), ""],
    ["นักวิจัยเฉลี่ยต่อโครงการ (คน)", round(researcher_n.mean(), 2), ""],
]

chapter4_summary = pd.DataFrame(summary_rows, columns=["ตัวชี้วัด", "จำนวน/ค่า", "ร้อยละ"])
display(chapter4_summary)
chapter4_summary.to_csv(OUTPUT_DIR/"chapter4_key_summary.csv", index=False, encoding="utf-8-sig")

print(f"บันทึกกราฟและตารางไว้ที่: {OUTPUT_DIR.resolve()}")


# หมายเหตุสำหรับการเขียนรายงาน

- ทุกกราฟควรใช้ชื่อเดียวกับหัวข้อในบทที่ 4 และใส่ `n (%)` ให้สอดคล้องกันทั้งเล่ม
- กราฟ Multiple response ต้องมีเชิงอรรถว่า **“โครงการหนึ่งสามารถอยู่ได้มากกว่า 1 ประเภท จึงรวมร้อยละเกิน 100% ได้”**
- Impact ต้องแยกถ้อยคำ **Actual / Expected** ในการเขียน narrative จาก Note ของแต่ละโครงการ ไม่ควรสรุปค่า 1 ทั้งหมดเป็นผลกระทบที่เกิดขึ้นจริง
- ส่วนผลประโยชน์วิชาการระดับ Outcome เรื่อง citation ยังไม่มีตัวแปรตรงในชีตนี้ จึงควรระบุว่า “ไม่มีข้อมูลสำหรับวิเคราะห์” หรือเติมจากแหล่งข้อมูลอื่นภายหลัง
- หากต้องการรวมคำตอบที่สะกดต่างกันแต่มีความหมายเดียวกัน ควรทำ **canonical mapping ที่ตรวจสอบโดยผู้วิจัย** ก่อนสร้างกราฟฉบับเผยแพร่
